In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import integrate
from ADSWIT.propagator  import *
from ADSWIT.model       import *
from ADSWIT.view        import *
from ADSWIT.utils       import *
from ADSWIT.survey      import *

## Basic Parameter

In [ ]:
device = "npu:0"         # Specify the CPU/GPU/NPU device
dtype = torch.float32
backend = ADFWI.set_backend(device, dtype=dtype)
ox,oz = 0,0
nz,nx = 100,100
dx,dz = 10,10
nt,dt = 2501,0.0002
nabc  = 100
f0    = 15
free_surface = False

## Velocity Model

In [ ]:
# velocity model 
# x = np.arange(0,nx*dx/1000,dx/1000)
# y = np.arange(0,nz*dz/1000,dz/1000)
# step = 1 #km
# vel_model = build_layer_model(x, y, step)
# vp  = vel_model['vp'].T * 1000
# rho = np.power(vp, 0.25) * 310
# vs  = vel_model['vs'].T * 1000

vp                      = np.ones((nz,nx))
vp[:30,:]               = 3724
vp[30:,:]               = 4640
# vp[99:,:]              = 5854
vs                      = np.ones((nz,nx))
vs[:30,:]               = 1944
vs[30:,:]               = 2583
# vs[99:,:]              = 3251
rho                     = np.ones((nz,nx))
rho[:30,:]              = 2450
rho[30:,:]              = 2490
# rho[99:,:]             = 2680

model = IsotropicElasticModel(ox,oz,nx,nz,dx,dz,
                              vp,vs,rho,
                              vp_grad=False,
                              vs_grad=False,
                              free_surface=free_surface,
                              abc_type="PML",abc_jerjan_alpha=0.007,
                              nabc=nabc
                    )
print(model.__repr__())
# model.save("./data/model.npz")

In [ ]:
model._plot_vp_vs_rho(figsize=(12,5),wspace=0.3,cbar_pad_fraction=0.18,cmap='jet_r')
model._plot_lam_mu(figsize=(12,5),wspace=0.15,cbar_pad_fraction=0.18,cmap='jet_r')

In [ ]:
model._plot("rho",cmap="jet_r")

GRTM
## Source and Receiver

In [ ]:
src_z = np.array([50]) 
src_x = np.array([50])
src_t,src_v = wavelet(nt,dt,f0,amp0=1)
# src_v = -integrate.cumtrapz(src_v, axis=-1, initial=0) #Integrate
src_v = -src_v
source = Source(nt=nt,dt=dt,f0=f0)
# Method 1
# source.add_sources(src_x=src_x,src_z=src_z,src_wavelet=src_v,src_type='mt',src_mt=np.array([[1,0,0],[0,1,0],[0,0,1]]))
# Method 2
for i in range(len(src_x)):
    source.add_source(src_x=src_x[i],src_z=src_z[i],src_wavelet=src_v,src_type="mt",src_mt=np.array([[0,0,1],[0,0,0],[0,0,0]]))

In [ ]:
if free_surface:
    np.savetxt("./data/free_surface_version3/wavelet.txt",src_v)
else:
    np.savetxt("./data/no_free_surface_version3_2layer/wavelet.txt",src_v)

In [ ]:
# receiver
rcv_z = np.array([1  for i in range(0,nx,1)])
rcv_x = np.array([j for j in range(0,nx,1)])
receiver = Receiver(nt=nt,dt=dt)
# method 1
# receiver.add_receivers(rcv_x=rcv_x,rcv_z=rcv_z,rcv_type='pr')
# method 2
for i in range(len(rcv_x)):
    receiver.add_receiver(rcv_x=rcv_x[i],rcv_z=rcv_z[i],rcv_type="pr")

In [ ]:
survey = Survey(source=source,receiver=receiver)
print(survey.__repr__())

In [ ]:
source.plot_wavelet()

In [ ]:
survey.plot(model.vs,cmap='jet_r')

In [ ]:
import json
if free_surface:
    np.savetxt("./data/free_surface_version3/vp.txt",vp)
    np.savetxt("./data/free_surface_version3/vs.txt",vs)
    np.savetxt("./data/free_surface_version3/rho.txt",rho)
else:
    np.savetxt("./data/no_free_surface_version3_2layer/vp.txt",vp)
    np.savetxt("./data/no_free_surface_version3_2layer/vs.txt",vs)
    np.savetxt("./data/no_free_surface_version3_2layer/rho.txt",rho)
params = {
    "dx":dx,
    "dz":dz,
    "nx":nx,
    "nz":nz,
    "nt":nt,
    "dt":dt,
    "pml":nabc,
    "f0":f0,
    "src":(np.hstack((src_x.reshape(-1,1),src_z.reshape(-1,1)))*dx).tolist(),
    "rcv":(np.hstack((rcv_x.reshape(-1,1),rcv_z.reshape(-1,1)))*dx).tolist(),
}
json_str = json.dumps(params)
if free_surface:
    with open("./data/free_surface_version3/params.json",'w') as json_file:
        json_file.write(json_str)
else:
    with open("./data/no_free_surface_version3_2layer/params.json",'w') as json_file:
        json_file.write(json_str)

## Propagator

In [ ]:
F = ElasticPropagator(model,survey)

In [ ]:
if model.abc_type == "PML":
    bcx = F.bcx
    bcz = F.bcz
    title_param = {'family':'Times New Roman','weight':'normal','size': 15}
    plot_bcx_bcz(bcx,bcz,dx=dx,dz=dz,wspace=0.25,title_param=title_param,cbar_height=0.04,cbar_pad_fraction=0.12)
else:
    damp = F.damp
    plot_damp(damp)

In [ ]:
record_waveform = F.forward(fd_order=4)
rcv_txx,rcv_tzz,rcv_txz,rcv_vx,rcv_vz = record_waveform["txx"],record_waveform["tzz"],record_waveform["txz"],record_waveform["vx"],record_waveform["vz"]
forward_wavefield_txx,forward_wavefield_tzz,forward_wavefield_txz,forward_wavefield_vx,forward_wavefield_vz = record_waveform["forward_wavefield_txx"],record_waveform["forward_wavefield_tzz"],record_waveform["forward_wavefield_txz"],record_waveform["forward_wavefield_vx"],record_waveform["forward_wavefield_vz"]

In [ ]:
d_obs = SeismicData(survey)
d_obs.record_data(record_waveform)
if free_surface:
    d_obs.save("./data/free_surface_version3/obs_data.npz")
else:
    d_obs.save("./data/no_free_surface_version3_2layer/obs_data.npz")

## Plotting Figure

In [ ]:
# waveform
shot = 0
plot_waveform2D(-(rcv_txx+rcv_tzz)[shot].T,figsize=(6,6),cmap='coolwarm')
plot_waveform_wiggle(-(rcv_txx+rcv_tzz)[shot],source.t,show=True)

In [ ]:
plt.figure()
plt.pcolormesh(-(rcv_tzz+rcv_txx)[0].detach().numpy())
plt.gca().invert_yaxis()
plt.show()

In [ ]:
shot = 0
trace = -1
plot_waveform_trace(-(rcv_tzz+rcv_txx),shot,trace,dt=dt)

In [ ]:
shot = 0
trace = -1
plot_waveform_trace(rcv_vx,shot,trace,dt=dt)

## devito

In [ ]:
devito_acoustic_p = np.load("/media/liufeng/a0b205ec-bfb3-473f-a6f0-0680c5da64ba/project/004_inversion/ADInversion/Others_work/Devito/devito/example_LF/data/elastic/2layer/shot0.npz")["data"]

In [ ]:
shot = 0
trace = 99

def normalize(data):
    return (data - data.mean())/(data.max() - data.min())

adswit_rcv_p = -(rcv_tzz+rcv_txx)

plt.figure()
plt.plot(normalize(adswit_rcv_p[shot,:,trace]),c='k',linestyle="--",label="ADSWIT")
plt.plot(normalize(-devito_acoustic_p[:,trace]),c='r',linestyle="--",label="Devito")
plt.legend()
plt.show()